### SQLite3 Magic Database for Querying

In [ ]:
import sqlite3
import pandas as pd

dfm3 = pd.read_csv('../data/dataMagic/cleanMagicIndPrices.csv')
# dfm3 = pd.read_pickle("../dataMagic/dfm3.pkl")

In [2]:
# Creates db if it doesn't already exist.
conn = sqlite3.connect("../database/magic.db")

#### Define tables for database.

In [3]:
cardsDF = dfm3[['name', 'setName', 'setCode', 'releaseDate', 'language', 'cardFinish', 'types', 'colors', 'rarity', 'gameAvailability', 'uuid']]
pricesDF = dfm3[['sourceDate', 'price', 'priceProvider', 'avgMarketPrice', 'providerListing', 'uuid']]

cardsDF.to_sql('cards', conn, index=False, if_exists='replace')
pricesDF.to_sql('prices', conn, index=False, if_exists='replace')

conn.commit()

In [4]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,cards
1,prices


#### Queries

##### How many unique cards are printed per set for the top 10 sets?

In [5]:
query1 = """
SELECT
    c.setname,
    COUNT(DISTINCT c.uuid) as numCards,
    strftime('%Y', c.releaseDate) AS releaseYear
FROM cards c
GROUP BY c.setname
ORDER BY numCards DESC
Limit 10;
"""

result1 = pd.read_sql(query1, conn)
result1

,setName,numCards,releaseYear
0,The List,5029,2020
1,Secret Lair Drop,2120,2019
2,Commander Legends: Battle for Baldur's Gate,952,2022
3,Commander Legends,712,2020
4,Ninth Edition,669,2005
5,Eighth Edition,667,2003
6,Seventh Edition,660,2001
7,Innistrad: Double Feature,633,2022
8,Unfinity,620,2022
9,"Warhammer 40,000",593,2022


##### What is the price vs. average market price for a particular card (Presence of the Master)?

In [6]:
query2 = """
SELECT DISTINCT
    c.name,
    c.setname,
    p.priceProvider,
    p.price,
    p.avgMarketPrice,
    c.cardFinish,
    c.uuid
FROM cards c
JOIN prices p ON c.uuid = p.uuid
WHERE c.name = 'Presence of the Master'
ORDER BY p.price DESC;
"""

result2 = pd.read_sql(query2, conn)
result2

,name,setName,priceProvider,price,avgMarketPrice,cardFinish,uuid
0,Presence of the Master,Legends,tcgplayer,10.53,10.03,normal,3c540848-d4ca-5441-a098-51a295e39aef
1,Presence of the Master,Legends,cardsphere,10.52,10.03,normal,3c540848-d4ca-5441-a098-51a295e39aef
2,Presence of the Master,Legends,cardkingdom,9.99,10.03,normal,3c540848-d4ca-5441-a098-51a295e39aef
3,Presence of the Master,Legends,manapool,9.09,10.03,normal,3c540848-d4ca-5441-a098-51a295e39aef
4,Presence of the Master,Urza's Saga,cardkingdom,0.59,0.44,normal,cd9ed8e9-3778-5e5c-907e-db5f41dbc215
5,Presence of the Master,Urza's Saga,cardsphere,0.44,0.44,normal,cd9ed8e9-3778-5e5c-907e-db5f41dbc215
6,Presence of the Master,Urza's Saga,tcgplayer,0.43,0.44,normal,cd9ed8e9-3778-5e5c-907e-db5f41dbc215
7,Presence of the Master,Urza's Saga,manapool,0.30,0.44,normal,cd9ed8e9-3778-5e5c-907e-db5f41dbc215


##### What is the average market price of each card rarity across the entire game?

In [7]:
query3 = """
SELECT DISTINCT
    c.rarity,
    AVG(p.avgMarketPrice) as avgPrice
FROM cards c
JOIN prices p ON c.uuid = p.uuid
WHERE c.rarity IN ('mythic', 'rare', 'uncommon', 'common')
GROUP BY c.rarity
ORDER BY avgPrice;
"""

result3 = pd.read_sql(query3, conn)
result3

,rarity,avgPrice
0,common,1.020627
1,uncommon,2.067605
2,rare,11.951334
3,mythic,12.867594


#### What top fifteen card types and colors hold the most value to consumers?

In [ ]:
query4 = """
SELECT DISTINCT
    c.types,
    c.colors,
    SUM(p.avgMarketPrice) as totalPrice
FROM cards c
JOIN prices p ON c.uuid = p.uuid
GROUP BY
    c.types,
    c.colors
ORDER BY totalPrice DESC
LIMIT 15;
"""

result4 = pd.read_sql(query4, conn)

# Store for visualization.
%store result4

result4

Stored 'result4' (DataFrame)


,types,colors,totalPrice
0,Artifact,C,2183060.20
1,Land,C,1781699.30
2,Creature,G,605459.16
3,Creature,B,572613.55
4,Creature,R,528688.49
5,Creature,W,484114.09
6,Creature,U,457401.25
7,Instant,U,412264.39
8,Sorcery,U,342819.94
9,Sorcery,B,319901.19


#### What are the top 15 most expensive green creatures by market price?

In [10]:
query5 = """
SELECT DISTINCT
    c.name,
    c.setName,
    c.types,
    c.colors,
    p.avgMarketPrice
FROM cards c
JOIN prices p ON c.uuid = p.uuid
WHERE c.types == 'Creature' AND c.colors == 'G'
ORDER BY p.avgMarketPrice DESC
LIMIT 15;
"""

result5 = pd.read_sql(query5, conn)
result5

,name,setName,types,colors,avgMarketPrice
0,Birds of Paradise,Limited Edition Alpha,Creature,G,3719.99
1,Gaea's Liege,Limited Edition Alpha,Creature,G,3626.07
2,Birds of Paradise,Seventh Edition,Creature,G,2999.99
3,Birds of Paradise,Limited Edition Beta,Creature,G,2977.47
4,Force of Nature,Limited Edition Alpha,Creature,G,1327.84
5,Verduran Enchantress,Limited Edition Alpha,Creature,G,1066.20
6,Elvish Archers,Limited Edition Alpha,Creature,G,828.99
7,Cockatrice,Limited Edition Alpha,Creature,G,658.73
8,Timber Wolves,Limited Edition Alpha,Creature,G,512.16
9,Verduran Enchantress,Limited Edition Beta,Creature,G,477.62
